# SPLADE (2021)
---
[[paper]](https://arxiv.org/pdf/2107.05720)<br>SPLADE = SParse Lexicalized and Aggregated DEcomposition<br>
Авторы: Thibault Formal, Patrick Gallinari, Stéphane Ayache (совместно с Sorbonne Université и Naver Labs Europe)

SPLADE — это **модель Dense Retrieval**, которая генерирует **разреженные (sparse), лексикализированные представления** для документов и запросов, используя один и тот же Transformer-энкодер.

### Контекст
В информационном поиске исторически доминировали два основных подхода:
1.  **Sparse Retrieval** (например, BM25 (1994)): основывается на лексическом совпадении ключевых слов. Эффективны для точного совпадения терминов и хорошо работают с инвертированными индексами, но страдают от проблемы "лексического разрыва" (lexical gap), когда семантически похожие, но лексически разные запросы или документы не находят друг друга.
2.  **Dense Retrieval** (например, DPR (2020), ColBERT (2020)): используют глубокие нейронные сети (обычно Transformer-энкодеры) для создания плотных (dense) векторных представлений (embeddings). Они отлично справляются с семантическим сходством, но теряют интерпретируемость и часто не могут использовать традиционные инвертированные индексы, что требует более сложных и ресурсоёмких ANN (Approximate Nearest Neighbor) индексов. Также они могут проигрывать Sparse Retrieval на запросах, требующих точного совпадения редких терминов.

### Идея метода
Основная идея SPLADE заключается в том, чтобы **объединить сильные стороны Sparse и Dense Retrieval**, создавая нейронную модель, которая генерирует **разреженные представления, напоминающие по структуре tf-idf векторы**, но при этом обученные улавливать семантическое сходство. Это позволяет использовать преимущества лексических методов (например, инвертированные индексы, интерпретируемость), одновременно извлекая выгоду из семантического понимания нейронных сетей.

### Постановка задачи
Задача — ранжирование коллекции документов $D = \{d_1, d_2, \dots, d_N\}$ относительно заданного запроса $Q$. Цель — найти $K$ наиболее релевантных документов.

### Существующие альтернативы
На момент появления SPLADE основными игроками были:
*   **BM25 (1994):** Алгоритм Sparse Retrieval, основанный на статистике терминов (term frequency, inverse document frequency) и длине документа. Архитектурно не использует нейронные сети, полагается на эвристические формулы. Его главное преимущество — скорость и интерпретируемость благодаря использованию инвертированных индексов. Главный недостаток — неспособность улавливать семантические связи.
*   **DPR (2020):** Модель Dense Retrieval, использующая два отдельных BERT-энкодера (один для запросов, другой для документов) для генерации плотных векторов фиксированной размерности (например, 768). Архитектурно — **двухбашенная модель (two-tower model)**. Обучается с помощью контрастивного обучения. Преимущество — высокая производительность на задачах, требующих глубокого семантического понимания. Недостатки — низкая интерпретируемость, необходимость ANN-индексов, меньшая эффективность при точном совпадении ключевых слов.
*   **ColBERT (2020):** Ещё одна Dense Retrieval модель, которая, в отличие от DPR, не сжимает весь документ/запрос в один плотный вектор, а генерирует отдельный эмбеддинг для каждого токена. Ранжирование происходит за счёт суммы максимальных сходств между эмбеддингами токенов запроса и документа (MaxSim). Архитектурно — **late interaction** модель. Пытается сохранить большую гранулярность, чем DPR, но все ещё требует сложных индексов.

SPLADE предлагает архитектурное отличие от обоих подходов, пытаясь преодолеть их слабости, сохраняя преимущества.

### Архитектура
Модель SPLADE использует один и тот же предварительно обученный **Transformer-энкодер** (например, BERT, RoBERTa) для кодирования как запросов, так и документов.

Ключевое отличие заключается в выходном слое:
1.  **Агрегация токенов:** Вместо использования только `[CLS]` токена для представления всего текста, как в большинстве Dense Retrieval моделей, SPLADE агрегирует выходы всех токенов. Для каждого токена $t_i$ в документе/запросе, Transformer-энкодер генерирует скрытое состояние $h_i$.
2.  **Лексикализированная проекция:** Над скрытыми состояниями всех токенов добавляется линейный слой, который проецирует их в пространство, размерность которого равна размеру словаря Transformer-модели. Это означает, что для каждого токена $t_i$ и для каждого слова $w_j$ в словаре будет получен скор, показывающий, насколько $t_i$ "активирует" $w_j$.
3.  **Активационная функция и агрегация:** К выходам этого линейного слоя применяется нелинейная функция **`ReLU`** для обеспечения неотрицательности, а затем **`log`** для масштабирования. Полученные значения для всех токенов входного текста суммируются для каждого слова в словаре.
    Это можно представить как:
    $d\_vector_w = \sum_{i=1}^{L} \max(0, \log(1 + \text{FFN}(h_i)[w]))$
    где $L$ — длина последовательности токенов, $h_i$ — выход энкодера для $i$-го токена, $\text{FFN}$ — Feed-Forward Network, и $[w]$ означает компоненту, соответствующую слову $w$ из словаря.
    Фактически, это означает, что каждый токен "голосует" за слова из словаря, присваивая им веса.
4.  **Разреженность:** Получившийся вектор имеет размерность, равную размеру словаря (например, 30522 для BERT-base). Большинство значений в этом векторе будут нулевыми или очень маленькими благодаря функции `ReLU` и регуляризации, что делает его **разреженным**. Ненулевые значения представляют собой "взвешенные термины", которые модель считает важными для данного документа/запроса, даже если эти термины явно не присутствуют в исходном тексте (например, "автомобиль" может активировать "машина").

### Алгоритм обучения
SPLADE обычно обучается на основе **самоконтролируемого обучения (self-supervised learning)** или **контрастивного обучения (contrastive learning)**, подобно другим нейронным моделям поиска. Авторы использовали вариант **Inverse Cloze Task (ICT)** (2017), которая хорошо показала себя в DPR.
1.  **Конструкция примеров:** Из коллекции документов случайным образом выбирается один параграф в качестве "запроса" (Q), а другой параграф из того же документа — как "положительный" документ ($D^+$). "Отрицательные" документы ($D^-$) берутся либо из других документов коллекции (in-batch negatives), либо из специально отобранных "трудных" отрицательных примеров (hard negatives), которые BM25 считает релевантными, но на самом деле они таковыми не являются.
2.  **Прямой проход:** Запрос $Q$, положительный $D^+$ и все отрицательные $D^-$ пропускаются через один и тот же Transformer-энкодер SPLADE для получения их разреженных векторных представлений.
3.  **Вычисление скора релевантности:** Скор релевантности между запросом и документом вычисляется как **точечное произведение (dot-product)** их разреженных векторов.
    $\text{score}(Q, D) = Q_{vector} \cdot D_{vector} = \sum_{w \in \text{Vocab}} Q_{vector}[w] \cdot D_{vector}[w]$
    Благодаря разреженности, это точечное произведение очень эффективно вычисляется, так как суммируются только общие ненулевые компоненты.
4.  **Функция потерь:** Используется **Negative Log Likelihood Loss** (также известная как Cross-Entropy Loss), как и в DPR. Цель — максимизировать скор положительных пар $(Q, D^+)$ и минимизировать скор отрицательных пар $(Q, D^-)$.
5.  **Регуляризация разреженности (Sparsity Regularization):** Это критически важный компонент. К общей функции потерь добавляется **L1-регуляризация** на выходы моделей. L1-регуляризация заставляет многие веса стремиться к нулю, что и обеспечивает разреженность векторов.
    $\text{Loss} = -\log \frac{\exp(\text{score}(Q, D^+))}{\sum_{D_i \in \{D^+, D^-\}} \exp(\text{score}(Q, D_i))} + \lambda \cdot (||Q_{vector}||_1 + ||D_{vector}||_1)$
    где $\lambda$ — гиперпараметр, контролирующий степень разреженности.

### Алгоритм инференса
1.  **Построение индекса:**
    *   Для каждого документа в коллекции: пропустить его через обученную модель SPLADE для получения разреженного вектора.
    *   Полученные разреженные векторы сохраняются в **инвертированном индексе**, аналогично тому, как индексируются документы для BM25. Для каждого уникального слова в словаре инвертированный индекс будет хранить список документов, в которых это слово (или его семантический эквивалент, активированный моделью) имеет ненулевой вес, а также сам этот вес.
2.  **Обработка запроса:**
    *   Заданный запрос $Q$ пропускается через ту же модель SPLADE для получения его разреженного вектора.
3.  **Поиск и ранжирование:**
    *   Используя разреженный вектор запроса и инвертированный индекс документов, вычисляется точечное произведение между запросом и всеми потенциально релевантными документами (только теми, которые имеют общие ненулевые компоненты).
    *   Документы ранжируются по убыванию полученного скора релевантности, и топ-K документов возвращаются.

### Результаты
SPLADE показал впечатляющие результаты, демонстрируя способность превосходить или быть наравне как с традиционными Sparse Retrieval методами (BM25), так и с передовыми Dense Retrieval моделями (DPR) на различных бенчмарках Information Retrieval.
*   На датасетах MS MARCO Passage Ranking и Natural Questions (используемых для оценки Question Answering и Information Retrieval), SPLADE **превосходит BM25 в среднем на 20-30% по метрике MRR (Mean Reciprocal Rank)**, что указывает на значительно лучшее понимание релевантности.
*   По сравнению с DPR, SPLADE достигает **сопоставимой или даже немного лучшей производительности (прирост 1-2% MRR)** на некоторых задачах, при этом предлагая более высокую интерпретируемость и возможность использования более эффективных инвертированных индексов.
*   Важным достижением является то, что SPLADE позволяет использовать инвертированные индексы, что делает его **значительно более эффективным в продакшене** для больших коллекций документов по сравнению с DPR, требующим дорогостоящих ANN-индексов. Это также позволяет легче инкрементально обновлять индекс.

## 📝 Критический анализ

```markdown
# SPLADE (2021)
---
[[paper]](https://arxiv.org/pdf/2107.05720)<br>SPLADE = SParse Lexicalized and Aggregated DEcomposition<br>
Авторы: Thibault Formal, Patrick Gallinari, Stéphane Ayache (Sorbonne Université и Naver Labs Europe)

SPLADE — это **модель Dense Retrieval**, генерирующая **разреженные, лексикализированные представления** для документов и запросов с использованием одного Transformer-энкодера.

### Контекст
В информационном поиске выделяются два подхода:
1. **Sparse Retrieval** (например, BM25 (1994)): основан на лексическом совпадении, но страдает от "лексического разрыва".
2. **Dense Retrieval** (например, DPR (2020), ColBERT (2020)): использует нейронные сети для семантического сходства, но требует сложных ANN индексов и может проигрывать в точности редких терминов.

### Идея
SPLADE объединяет преимущества Sparse и Dense Retrieval, создавая разреженные представления, напоминающие tf-idf векторы, но обученные улавливать семантическое сходство. Это позволяет использовать инвертированные индексы и интерпретируемость.

### Постановка задачи
Ранжирование коллекции документов $D = \{d_1, d_2, \dots, d_N\}$ относительно запроса $Q$ для нахождения $K$ наиболее релевантных документов.

### Архитектура
Модель SPLADE использует один Transformer-энкодер для кодирования запросов и документов. Ключевые моменты:
1. **Агрегация токенов:** Используются все токены, а не только `[CLS]`.
2. **Лексикализированная проекция:** Линейный слой проецирует скрытые состояния в пространство словаря.
3. **Активация и агрегация:** Применяются `ReLU` и `log`, суммируя значения для каждого слова.
4. **Разреженность:** Вектор имеет размерность словаря, большинство значений нулевые, что делает его разреженным.

### Алгоритм обучения
Обучение на основе **контрастивного обучения** с использованием **Inverse Cloze Task (ICT)**:
1. **Конструкция примеров:** Используются положительные и отрицательные примеры.
2. **Прямой проход:** Получение разреженных векторов.
3. **Скор релевантности:** Вычисляется как **точечное произведение** векторов.
4. **Функция потерь:** Используется **Negative Log Likelihood Loss** с L1-регуляризацией для разреженности.

### Алгоритм инференса
1. **Построение индекса:** Разреженные векторы сохраняются в инвертированном индексе.
2. **Обработка запроса:** Запрос пропускается через модель для получения вектора.
3. **Поиск и ранжирование:** Вычисляется точечное произведение с документами, ранжируются по релевантности.

### Результаты
SPLADE превосходит BM25 на 20-30% по MRR и достигает сопоставимой производительности с DPR, предлагая более высокую интерпретируемость и эффективность в продакшене благодаря инвертированным индексам.

<img src="img/img.png" width=500>
```

## 💻 Пример кода

Иллюстративный Python пример, демонстрирующий основные концепции:

In [ ]:
# Пример реализации основных концепций SPLADE на Python.
# Мы будем использовать Hugging Face Transformers для работы с BERT и PyTorch для создания модели.

import torch
import torch.nn as nn
from transformers import BertTokenizer, BertModel

# Инициализация токенизатора и модели BERT
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
bert_model = BertModel.from_pretrained('bert-base-uncased')

# Пример текста запроса и документа
query_text = "What is the capital of France?"
document_text = "Paris is the capital city of France."

# Токенизация текста
query_tokens = tokenizer(query_text, return_tensors='pt')
document_tokens = tokenizer(document_text, return_tensors='pt')

# Получение скрытых состояний из BERT
query_hidden_states = bert_model(**query_tokens).last_hidden_state
document_hidden_states = bert_model(**document_tokens).last_hidden_state

# SPLADE: Лексикализированная проекция
class SPLADEProjection(nn.Module):
    def __init__(self, vocab_size):
        super(SPLADEProjection, self).__init__()
        # Линейный слой для проекции скрытых состояний в пространство словаря
        self.projection = nn.Linear(bert_model.config.hidden_size, vocab_size)

    def forward(self, hidden_states):
        # Применение линейного слоя
        logits = self.projection(hidden_states)
        # Применение ReLU и логарифма
        activated_logits = torch.log(1 + torch.relu(logits))
        # Агрегация по токенам (суммирование)
        aggregated_vector = torch.sum(activated_logits, dim=1)
        return aggregated_vector

# Инициализация SPLADE проекции
vocab_size = len(tokenizer.vocab)
splade_projection = SPLADEProjection(vocab_size)

# Получение разреженных векторов для запроса и документа
query_vector = splade_projection(query_hidden_states)
document_vector = splade_projection(document_hidden_states)

# Вычисление скора релевантности через точечное произведение
relevance_score = torch.dot(query_vector.squeeze(), document_vector.squeeze())

print(f"Relevance Score: {relevance_score.item()}")

# Пример регуляризации разреженности
l1_regularization = torch.norm(query_vector, p=1) + torch.norm(document_vector, p=1)
print(f"L1 Regularization Term: {l1_regularization.item()}")

# В реальной задаче обучения, к функции потерь добавляется L1-регуляризация для обеспечения разреженности
# loss = original_loss + lambda * l1_regularization
```

### Комментарии к коду:
1. **Токенизация и получение скрытых состояний:** Используем BERT для получения скрытых состояний токенов запроса и документа.
2. **Лексикализированная проекция:** Создаем линейный слой, который проецирует скрытые состояния в пространство размером с размер словаря. Применяем `ReLU` и `log` для активации и масштабирования.
3. **Агрегация:** Суммируем активированные логиты по всем токенам, чтобы получить разреженное представление.
4. **Вычисление скора релевантности:** Используем точечное произведение разреженных векторов запроса и документа.
5. **Регуляризация разреженности:** Пример вычисления L1-регуляризации, которая добавляется к функции потерь для обеспечения разреженности векторов.

Этот код иллюстрирует основные архитектурные особенности SPLADE, такие как лексикализированная проекция и использование разреженных векторов, которые отличают его от других методов информационного поиска.